In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.table("retail.retail_etl.bronze_customers")

display(df)

In [0]:
df_clean = (
    df
    .dropDuplicates(["customer_id"])
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("country", upper(col("country")))
    .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))
    .filter(col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$"))
    .withColumn("created_date", to_date(col("created_date")))
)

display(df_clean)

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail.retail_etl.silver_customers")

In [0]:
%sql
SELECT COUNT(*)
FROM retail.retail_etl.silver_customers;